# 01 – PLAsTiCC Data Exploration

Prerequisite: run `python scripts/prepare_data.py` first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data/processed/plasticc")
train = np.load(DATA_DIR / "train.npz")
label_map = np.load(DATA_DIR / "label_map.npy", allow_pickle=True).item()
idx_to_class = {v: k for k, v in label_map.items()}

lc = train["light_curves"]
labels = train["labels"]
print(f"Light curves shape: {lc.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Classes: {sorted(label_map.keys())}")

## Class Distribution

In [ ]:
from src.data.plasticc import CLASS_MAP

unique, counts = np.unique(labels, return_counts=True)
class_names = [CLASS_MAP.get(idx_to_class[u], str(u)) for u in unique]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(class_names, counts)
ax.set_xlabel("Transient class")
ax.set_ylabel("Count")
ax.set_title("PLAsTiCC Training Set – Class Distribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Example Light Curves (all 6 bands)

In [ ]:
PASSBAND_NAMES = {0: 'u', 1: 'g', 2: 'r', 3: 'i', 4: 'z', 5: 'Y'}
COLORS = ['purple', 'green', 'red', 'orange', 'brown', 'black']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
# Pick one sample per class (first 6 classes)
for ax_idx, cls_idx in enumerate(unique[:6]):
    ax = axes.flat[ax_idx]
    sample_idx = np.where(labels == cls_idx)[0][0]
    for pb in range(6):
        flux = lc[sample_idx, pb, :, 0]
        nonzero = np.nonzero(flux)[0]
        if len(nonzero) > 0:
            ax.plot(flux[:nonzero[-1]+1], color=COLORS[pb], label=PASSBAND_NAMES[pb], alpha=0.7)
    ax.set_title(CLASS_MAP.get(idx_to_class[cls_idx], str(cls_idx)))
    ax.legend(fontsize=7)
plt.suptitle("Example Light Curves")
plt.tight_layout()
plt.show()

## GAF Image Examples

In [ ]:
from src.data.augmentation import light_curve_to_gaf

sample_idx = 0
fig, axes = plt.subplots(1, 6, figsize=(20, 3))
for pb in range(6):
    flux = lc[sample_idx, pb, :, 0]
    nonzero = np.nonzero(flux)[0]
    if len(nonzero) > 2:
        flux_trim = flux[:nonzero[-1]+1]
    else:
        flux_trim = flux[:10]
    gaf = light_curve_to_gaf(flux_trim, image_size=64, method='gasf')
    axes[pb].imshow(gaf, cmap='viridis', aspect='auto')
    axes[pb].set_title(f"Band {PASSBAND_NAMES[pb]}")
    axes[pb].axis('off')
plt.suptitle(f"GASF images – class {CLASS_MAP.get(idx_to_class[labels[sample_idx]], '?')}")
plt.tight_layout()
plt.show()

## DataLoader Test

In [ ]:
from torch.utils.data import DataLoader
from src.data.plasticc import PLAsTiCCDataset

# Test timeseries loader
ds = PLAsTiCCDataset(DATA_DIR, split='train', representation='timeseries')
x, y = ds[0]
print(f"Timeseries sample: x.shape={x.shape}, y={y}")

dl = DataLoader(ds, batch_size=16, shuffle=True)
batch_x, batch_y = next(iter(dl))
print(f"Batch: x.shape={batch_x.shape}, y.shape={batch_y.shape}")